# Production hardening for Amazon Bedrock Mantle

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Everything to get right before a mantle workload carries real traffic. Each
section is a control you can verify, not advice you have to take on trust.

## What this notebook covers
- Credential lifecycle (short-term tokens, refresh, SigV4 (AWS Signature Version 4))
- Retries, timeouts, and the failure modes that actually occur
- Quota reality: no RPM, separate in/out TPM (tokens per minute), mostly unpublished
- Data retention posture and ZDR (zero data retention)
- Cost attribution with Projects
- Observability, including the namespace that catches people out
- Defensive output handling
- Guardrails, and why they are not a mantle parameter
- A pre-launch checklist you can run

## Self-contained, but see also
- **Auth and the three paths** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Governance and retention** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas and tiers** → `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, boto3, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `list_models` | the `bedrock-mantle` model inventory for a Region |
| `parse_json_lenient` | parses the first complete JSON object out of model output, repairing truncated braces |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `response_text` | assistant text from a Responses API payload |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import concurrent.futures as cf
import json
import random
import sys
import time
import urllib.error
import urllib.request

sys.path.insert(0, "../_shared")
from bedrock import (
    err,
    list_models,
    parse_json_lenient,
    post,
    resolve_runtime_id,
    response_text,
    safe_print,
)

REGION = "us-east-1"
MODEL = "google.gemma-4-31b"  # present in all four mantle Regions
PREFIX = "/openai/v1"
print("region:", REGION, "| model:", MODEL)

region: us-east-1 | model: google.gemma-4-31b


## 1. Credentials — mint short, refresh often, never store

Short-term Bedrock API keys are presigned SigV4 requests: they *are* IAM. They
expire within 12 hours, **cannot be refreshed**, and are Region-pinned.

Three rules:
1. Mint in-process from the ambient role. Do not put a 15-minute secret in a
   secrets manager.
2. Keep the TTL short. The token is your role until it expires.
3. Never ship long-term keys — they create a static IAM user credential.

In [2]:
import threading
from datetime import datetime, timedelta, timezone

from aws_bedrock_token_generator import provide_token


class TokenProvider:
    """Thread-safe short-term token cache with early refresh."""

    def __init__(self, region=REGION, ttl=timedelta(minutes=15), skew_s=120):
        self.region, self.ttl, self.skew_s = region, ttl, skew_s
        self._token = None
        self._expires_at = None
        self._lock = threading.Lock()
        self.mints = 0

    def get(self) -> str:
        now = datetime.now(timezone.utc)
        with self._lock:  # avoid a thundering herd
            if self._token and self._expires_at and now < self._expires_at:
                return self._token
            self._token = provide_token(region=self.region, expiry=self.ttl)
            self._expires_at = now + self.ttl - timedelta(seconds=self.skew_s)
            self.mints += 1
            return self._token


tokens = TokenProvider()
with cf.ThreadPoolExecutor(max_workers=8) as pool:
    values = list(pool.map(lambda _: tokens.get(), range(8)))
print(
    f"8 concurrent callers -> {tokens.mints} mint(s), "
    f"all identical: {len(set(values)) == 1}"
)
print("expires around:", tokens._expires_at.isoformat(timespec="seconds"))

8 concurrent callers -> 1 mint(s), all identical: True
expires around: 2026-08-20T06:11:31+00:00


In [3]:
# For roles with no need of a bearer token at all, SigV4-sign directly.
# Signing name is "bedrock". SigV4 callers do NOT need CallWithBearerToken.
import boto3
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from botocore.httpsession import URLLib3Session

body = json.dumps({"model": MODEL, "input": "Reply OK", "max_output_tokens": 16})
creds = boto3.Session().get_credentials().get_frozen_credentials()
request = AWSRequest(
    method="POST",
    url=f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}/responses",
    data=body,
    headers={"Content-Type": "application/json"},
)
SigV4Auth(creds, "bedrock", REGION).add_auth(request)
resp = URLLib3Session(timeout=60).send(request.prepare())
print("SigV4 path ->", resp.status_code)

SigV4 path -> 200


## 2. Retries, timeouts, and what actually fails

Mantle has **no RPM quota**; throttling is token-based, and most models have no
published TPM at all — capacity is internal fair-share. So `429` and `5xx` are
ordinary operating conditions, not exceptions.

Equally important: a **wrong path can stall** rather than return an error. Without
a client timeout, a retry loop turns that into a multi-minute hang.

In [4]:
TRANSIENT = {429, 500, 502, 503, 504}


def open_https(req, timeout: int):
    """urlopen restricted to HTTPS.

    urllib also honours file://, ftp:// and data:// . These URLs are all built
    from literals, but a client that ever takes a URL from data would let those
    schemes read local files, so the guard belongs in the helper (CWE-22).
    """
    if not req.full_url.startswith("https://"):
        raise ValueError(f"refusing non-HTTPS URL: {req.full_url[:60]}")
    # nosemgrep: dynamic-urllib-use-detected - scheme verified https above
    return urllib.request.urlopen(req, timeout=timeout)  # nosec B310  # noqa: S310


def resilient_call(path, body, *, region=REGION, headers=None, attempts=5, timeout=60):
    """Retry transient failures; fail fast on client errors; always time out."""
    url = f"https://bedrock-mantle.{region}.api.aws{path}"
    payload = json.dumps(body).encode()
    for attempt in range(attempts):
        hdrs = {
            "Authorization": f"Bearer {tokens.get()}",
            "Content-Type": "application/json",
            **(headers or {}),
        }
        req = urllib.request.Request(url, data=payload, headers=hdrs, method="POST")
        try:
            with open_https(req, timeout=timeout) as response:
                return response.status, json.loads(response.read())
        except urllib.error.HTTPError as exc:
            if exc.code in TRANSIENT and attempt < attempts - 1:
                time.sleep(
                    # Jitter spreads retries; not a security decision.
                    min(2**attempt, 16)
                    + random.random()  # nosec B311
                )
                continue
            return exc.code, json.loads(exc.read() or b"{}")
        except Exception as exc:  # timeout, connection reset
            if attempt < attempts - 1:
                time.sleep(
                    # Jitter spreads retries; not a security decision.
                    min(2**attempt, 16)
                    + random.random()  # nosec B311
                )
                continue
            return -1, {"error": {"message": f"{type(exc).__name__}"}}
    return -1, {"error": {"message": "retries exhausted"}}


code, data = resilient_call(
    f"{PREFIX}/responses",
    {"model": MODEL, "input": "Reply OK", "max_output_tokens": 16},
)
print("healthy call ->", code, repr(response_text(data)[:30]))

healthy call -> 200 'OK'


In [5]:
# A permanent 400 must not be retried — measure that it fails fast.
#
# The invalid input here is `max_output_tokens` below the documented minimum of 16.
# That choice matters: an earlier version of this cell sent `top_p` to Gemma 4,
# which was a 400 when written and is a 200 now, so the cell stopped demonstrating
# anything. Pick an invariant to violate, not a per-model restriction.
started = time.perf_counter()
code, data = resilient_call(
    f"{PREFIX}/responses",
    {"model": MODEL, "input": "Hi", "max_output_tokens": 8},
)
print(
    f"invalid param -> HTTP {code} in {time.perf_counter() - started:.2f}s "
    f"(no retries — correct)"
)
print("message:", err(data)[:90])

invalid param -> HTTP 400 in 0.71s (no retries — correct)
message: Invalid 'max_output_tokens': integer below minimum value. Expected a value >= 16, but got 


In [6]:
# The stall case, contained by a short timeout. Grok on the bare /v1 responses path
# does not answer at all.
started = time.perf_counter()
code, data = post(
    "/v1/responses",
    {"model": "xai.grok-4.3", "input": "Hi", "max_output_tokens": 16},
    region=REGION,
    attempts=1,
    timeout=20,
)
print(
    f"stalling path -> {code} in {time.perf_counter() - started:.1f}s "
    f"(bounded by the timeout, not by the server)"
)

stalling path -> -1 in 20.5s (bounded by the timeout, not by the server)


## 3. Concurrency: ramp, don't spike

A cold start from zero to peak concurrency gets shed. Step up gradually.

In [7]:
def one_call(i):
    code, _ = resilient_call(
        f"{PREFIX}/responses",
        {"model": MODEL, "input": f"Say OK ({i})", "max_output_tokens": 16},
    )
    return code


for concurrency in (1, 4, 8):
    started = time.perf_counter()
    with cf.ThreadPoolExecutor(max_workers=concurrency) as pool:
        codes = list(pool.map(one_call, range(concurrency)))
    elapsed = time.perf_counter() - started
    ok = sum(1 for c in codes if c == 200)
    print(f"concurrency {concurrency:2} -> {ok}/{concurrency} ok in {elapsed:5.2f}s")

print("\nIn production, step concurrency up over minutes and keep the backoff loop.")

concurrency  1 -> 1/1 ok in  1.05s


concurrency  4 -> 4/4 ok in  0.97s


concurrency  8 -> 8/8 ok in  1.00s

In production, step concurrency up over minutes and keep the backoff loop.


## 4. Data retention posture

Two independent controls. Get both explicit.

| Control | Scope | Default |
|---|---|---|
| `store` | per request (Responses) | **`true`** — 30-day retention |
| `data_retention.mode` | account / project / model | `inherit` |

Modes: `default`, `none` (zero data retention), `provider_data_share`, `inherit`.

In [8]:
code, retention = post("/v1/data_retention", None, region=REGION, method="GET")
print("account retention:", retention)

# store defaults to true — verify by omitting it.
code, implicit = post(
    f"{PREFIX}/responses",
    {"model": MODEL, "input": "Reply OK", "max_output_tokens": 16},
    region=REGION,
)
code, explicit = post(
    f"{PREFIX}/responses",
    {"model": MODEL, "input": "Reply OK", "max_output_tokens": 16, "store": False},
    region=REGION,
)
print(f"store omitted   -> store={implicit.get('store')}  (retained 30 days)")
print(f"store=False     -> store={explicit.get('store')}")

account retention: {'mode': 'inherit'}


store omitted   -> store=True  (retained 30 days)
store=False     -> store=False


In [9]:
# A retrievable response proves retention is real.
retrievable = post(
    f"{PREFIX}/responses",
    {
        "model": MODEL,
        "input": "Remember: alpha.",
        "max_output_tokens": 16,
        "store": True,
    },
    region=REGION,
)[1]
code, fetched = post(
    f"{PREFIX}/responses/{retrievable['id']}", None, region=REGION, method="GET"
)
print(f"GET stored response -> {code} (status={fetched.get('status')})")
code, _ = post(
    f"{PREFIX}/responses/{retrievable['id']}", None, region=REGION, method="DELETE"
)
print(f"DELETE it           -> {code}")

GET stored response -> 200 (status=completed)


DELETE it           -> 200


In [10]:
# Some models are gated by retention mode. Check before you debug your request.
for mid in (MODEL, "anthropic.claude-fable-5"):
    code, info = post(f"/v1/models/{mid}", None, region=REGION, method="GET")
    if code != 200:
        print(f"{mid:34} GET -> {code}")
        continue
    dr = info.get("data_retention", {})
    print(
        f"{mid:34} status={info.get('status'):12} "
        f"allowed_modes={dr.get('allowed_modes')}"
    )
    if info.get("status_reason"):
        print(f"      reason: {info['status_reason'][:100]}")

google.gemma-4-31b                 status=available    allowed_modes=['none', 'default', 'provider_data_share']


anthropic.claude-fable-5           status=unavailable  allowed_modes=['provider_data_share']
      reason: This model is not available under data retention mode 'default'.


## 5. Cost attribution with Projects

Every production workload should run under its own tagged project. Untagged usage
is impossible to allocate later.

In [11]:
code, project = post(
    "/v1/organization/projects",
    {
        "name": "hardening-demo",
        "tags": {
            "Application": "HardeningDemo",
            "Environment": "Demo",
            "Owner": "PlatformTeam",
            "CostCenter": "0000",
        },
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id, "| tags:", project.get("tags"))

# The attribution header differs by API — a classic mistake.
for label, path, body, header in [
    (
        "Responses",
        f"{PREFIX}/responses",
        {"model": MODEL, "input": "Reply OK", "max_output_tokens": 16},
        {"OpenAI-Project": project_id},
    ),
    (
        "ChatCompletions",
        "/v1/chat/completions",
        {
            "model": "qwen.qwen3-32b",
            "messages": [{"role": "user", "content": "Reply OK"}],
            "max_tokens": 16,
        },
        {"OpenAI-Project": project_id},
    ),
    (
        "Messages",
        "/anthropic/v1/messages",
        {
            "model": "anthropic.claude-haiku-4-5",
            "max_tokens": 16,
            "messages": [{"role": "user", "content": "Reply OK"}],
        },
        {"anthropic-workspace": project_id, "anthropic-version": "2023-06-01"},
    ),
]:
    code, _ = post(path, body, region=REGION, headers=header)
    used = [k for k in header if k != "anthropic-version"][0]
    print(f"  {label:16} via {used:22} -> {code}")

project: 200 proj_a7y35wsx... | tags: {'CostCenter': '0000', 'Owner': 'PlatformTeam', 'Application': 'HardeningDemo', 'Environment': 'Demo'}


  Responses        via OpenAI-Project         -> 200


  ChatCompletions  via OpenAI-Project         -> 200


  Messages         via anthropic-workspace    -> 200


## 6. Observability — the namespace trap

Mantle publishes to **`AWS/BedrockMantle`**, not `AWS/Bedrock`. A dashboard built
on the wrong namespace shows **zero errors during an incident**. And the only error
metric is `InferenceClientErrors` (4xx) — there is **no server-error metric**, so
503s must come from your own telemetry.

In [12]:
cw = boto3.client("cloudwatch", region_name=REGION)
for namespace in ("AWS/BedrockMantle", "AWS/Bedrock"):
    metrics = sorted(
        {
            m["MetricName"]
            for m in cw.list_metrics(Namespace=namespace).get("Metrics", [])
        }
    )
    print(f"{namespace:22} {len(metrics):2} metrics: {', '.join(metrics[:6])}")
print("\nNote the absence of any 5xx/server-error metric in AWS/BedrockMantle.")

AWS/BedrockMantle       8 metrics: BurnDownConsumed, EquivalentReservationUnits, InferenceClientErrors, Inferences, InputTokens, OutputTokens


AWS/Bedrock            11 metrics: CacheReadInputTokenCount, CacheWriteInputTokenCount, EstimatedTPMQuotaUsage, InputTokenCount, InvocationClientErrors, InvocationLatency

Note the absence of any 5xx/server-error metric in AWS/BedrockMantle.


In [13]:
# Because 5xx is invisible server-side, record it client-side.
class CallMetrics:
    """Minimal client-side telemetry — the only place 5xx is visible."""

    def __init__(self):
        self.counts = {
            "ok": 0,
            "throttled": 0,
            "server_error": 0,
            "client_error": 0,
            "timeout": 0,
        }
        self.latencies = []

    def record(self, code, seconds):
        self.latencies.append(seconds)
        if code == 200:
            self.counts["ok"] += 1
        elif code == 429:
            self.counts["throttled"] += 1
        elif code == -1:
            self.counts["timeout"] += 1
        elif 500 <= code < 600:
            self.counts["server_error"] += 1
        else:
            self.counts["client_error"] += 1

    def report(self):
        if not self.latencies:
            return "no calls"
        ordered = sorted(self.latencies)
        p50 = ordered[len(ordered) // 2]
        p95 = ordered[min(int(len(ordered) * 0.95), len(ordered) - 1)]
        return (
            f"{self.counts} | p50={p50:.2f}s p95={p95:.2f}s " f"n={len(self.latencies)}"
        )


metrics = CallMetrics()
for i in range(6):
    started = time.perf_counter()
    code, _ = resilient_call(
        f"{PREFIX}/responses",
        {"model": MODEL, "input": f"Reply OK ({i})", "max_output_tokens": 16},
        headers={"OpenAI-Project": project_id},
    )
    metrics.record(code, time.perf_counter() - started)
print(metrics.report())

{'ok': 6, 'throttled': 0, 'server_error': 0, 'client_error': 0, 'timeout': 0} | p50=1.27s p95=1.94s n=6


**CloudTrail:** mantle inference is a **data event** — off by default, and billed
extra when enabled. Management events (creating projects) appear normally. Note
that short-term key *generation* is client-side and never logged.

## 7. Defensive output handling

HTTP 200 does not mean you got what you asked for. Three real cases:

1. **Truncation** — `finish_reason="length"` with empty content.
2. **Trailing characters** after valid JSON, even in strict mode.
3. **Ignored constraints** — a forced tool choice that returns prose.

In [14]:
SCHEMA = {
    "type": "object",
    "properties": {"language": {"type": "string"}, "typed": {"type": "boolean"}},
    "required": ["language", "typed"],
    "additionalProperties": False,
}


def safe_structured(
    model, prompt, schema, *, prefix=PREFIX, max_output_tokens=600, attempts=3
):
    """Structured output with truncation detection, lenient parsing and retries."""
    budget = max_output_tokens
    for attempt in range(attempts):
        code, data = post(
            f"{prefix}/responses",
            {
                "model": model,
                "input": prompt,
                "max_output_tokens": budget,
                "text": {
                    "format": {
                        "type": "json_schema",
                        "name": "out",
                        "schema": schema,
                        "strict": True,
                    }
                },
                "store": False,
            },
            region=REGION,
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        text = response_text(data)
        if not text.strip():
            budget *= 3  # truncated before emitting anything
            print(f"   attempt {attempt + 1}: empty output, raising budget to {budget}")
            continue
        try:
            parsed = parse_json_lenient(text)  # tolerates trailing characters
        except ValueError as exc:
            print(f"   attempt {attempt + 1}: unparseable ({exc})")
            continue
        missing = set(schema["required"]) - set(parsed)
        if missing:
            print(f"   attempt {attempt + 1}: missing keys {missing}")
            continue
        return parsed
    raise RuntimeError("no valid structured output after retries")


print("structured with guards:")
print("  ", safe_structured(MODEL, "Describe the Go programming language.", SCHEMA))

structured with guards:


   {'language': 'Go', 'typed': True}


In [15]:
# Show the trailing-character problem that motivates lenient parsing.
raw_samples = [
    '{"language":"Go","typed":true}',
    '{"language":"Go","typed":true}\n}',
    '{"language":"Go","typed":true}\nextra text',
]
for sample in raw_samples:
    try:
        json.loads(sample)
        verdict = "json.loads OK"
    except json.JSONDecodeError:
        verdict = "json.loads FAILS"
    print(f"  {verdict:18} | lenient -> {parse_json_lenient(sample)} | {sample!r}")

  json.loads OK      | lenient -> {'language': 'Go', 'typed': True} | '{"language":"Go","typed":true}'
  json.loads FAILS   | lenient -> {'language': 'Go', 'typed': True} | '{"language":"Go","typed":true}\n}'
  json.loads FAILS   | lenient -> {'language': 'Go', 'typed': True} | '{"language":"Go","typed":true}\nextra text'


## 8. Region and model availability as a pre-flight check

Verify at startup that every model you depend on exists in your Region. Failing
here is much better than failing on first traffic.

In [16]:
REQUIRED = [MODEL, "qwen.qwen3-32b", "anthropic.claude-haiku-4-5", "openai.gpt-5.6-sol"]


def preflight(required, region=REGION):
    """Fail fast at startup if a dependency is missing from this Region."""
    available = set(list_models(region))
    missing = [m for m in required if m not in available]
    return {
        "region": region,
        "ok": not missing,
        "missing": missing,
        "available_count": len(available),
    }


for region in ("us-east-1", "eu-central-1"):
    result = preflight(REQUIRED, region)
    status = "PASS" if result["ok"] else "FAIL"
    print(
        f"{region:14} {status}  ({result['available_count']} models) "
        f"missing={result['missing']}"
    )

us-east-1      PASS  (55 models) missing=[]


eu-central-1   FAIL  (33 models) missing=['anthropic.claude-haiku-4-5', 'openai.gpt-5.6-sol']


## 9. A production client, assembled

Every control from above in one place.

In [17]:
class ProductionClient:
    """Hardened bedrock-mantle client.

    Routing is the part that used to be wrong here. An earlier version computed the
    right path *prefix* per model and then always POSTed `{prefix}/responses`, so it
    400ed for every Claude model and every Chat-Completions-only family — i.e. for
    most of the catalogue. Resolve the **surface** as well as the prefix.

    Sampling is handled by dropping what the service names in a 400 rather than by
    carrying a per-model table. Those tables have gone stale twice in this
    collection's lifetime; the error message has not moved.
    """

    CHAT_ONLY = ("qwen.", "deepseek.", "zai.", "minimax.", "moonshotai.",
                 "mistral.", "nvidia.", "writer.", "openai.gpt-oss-safeguard",
                 "google.gemma-3")
    COMPLETION_TOKENS = ("openai.gpt-5.6",)
    TIERED = ("openai.gpt-oss", "google.gemma-", "xai.", "qwen.", "deepseek.",
              "zai.", "minimax.", "moonshotai.", "mistral.", "nvidia.", "writer.")
    TUNABLE = ("temperature", "top_p", "service_tier")

    def __init__(self, model, region=REGION, project=None, tier="default"):
        self.model, self.region, self.project = model, region, project
        self.tokens = TokenProvider(region=region)
        self.metrics = CallMetrics()
        if model.startswith("anthropic."):
            self.prefix, self.surface = "/anthropic/v1", "messages"
        elif model.startswith(self.CHAT_ONLY):
            self.prefix = "/openai/v1" if model.startswith(
                ("google.gemma-4", "openai.gpt-5", "xai.")) else "/v1"
            self.surface = "chat"
        else:
            self.prefix = "/openai/v1" if model.startswith(
                ("google.gemma-4", "openai.gpt-5", "xai.")) else "/v1"
            self.surface = "responses"
        self.tier = (tier if (tier == "default" or model.startswith(self.TIERED))
                     else "default")

    def _budget_field(self):
        if self.surface == "responses":
            return "max_output_tokens"
        if self.surface == "chat" and self.model.startswith(self.COMPLETION_TOKENS):
            return "max_completion_tokens"
        return "max_tokens"

    def _build(self, prompt, budget, schema, sampling):
        body = {"model": self.model, self._budget_field(): max(16, budget),
                **sampling}
        if self.surface == "messages":
            body["messages"] = [{"role": "user", "content": prompt}]
            return f"{self.prefix}/messages", body
        body["service_tier"] = self.tier
        if self.surface == "chat":
            body["messages"] = [{"role": "user", "content": prompt}]
            if schema:
                body["response_format"] = {
                    "type": "json_schema",
                    "json_schema": {"name": "out", "strict": True, "schema": schema},
                }
            return f"{self.prefix}/chat/completions", body
        body["input"] = prompt
        body["store"] = False  # explicit: no 30-day retention
        if schema:
            body["text"] = {"format": {"type": "json_schema", "name": "out",
                                       "schema": schema, "strict": True}}
        return f"{self.prefix}/responses", body

    @staticmethod
    def _refused_param(message):
        """Which tunable the service just named, if any. See 02-migrating §7."""
        import re

        quoted = re.findall(r"[\'`\"]([a-z_]+)[\'`\"]", message or "")
        for name in quoted:
            if name in ProductionClient.TUNABLE:
                return name
        if "API" not in (message or ""):
            for name in ProductionClient.TUNABLE:
                if name in (message or ""):
                    return name
        return None

    def _extract(self, data):
        if self.surface == "messages":
            return "".join(b.get("text", "") for b in data.get("content", [])
                           if b.get("type") == "text")
        if self.surface == "chat":
            return (data.get("choices") or [{}])[0].get("message", {}).get(
                "content") or ""
        return response_text(data)

    def ask(self, prompt, *, max_output_tokens=512, temperature=None, top_p=None,
            schema=None):
        sampling = {}
        if temperature is not None:
            sampling["temperature"] = temperature
        if top_p is not None:
            sampling["top_p"] = top_p

        headers = {}
        if self.project:
            key = ("anthropic-workspace" if self.surface == "messages"
                   else "OpenAI-Project")
            headers[key] = self.project
        if self.surface == "messages":
            headers["anthropic-version"] = "2023-06-01"

        for _ in range(len(self.TUNABLE) + 1):
            path, body = self._build(prompt, max_output_tokens, schema, sampling)
            started = time.perf_counter()
            code, data = post(path, body, region=self.region,
                              headers=headers or None, timeout=90)
            self.metrics.record(code, time.perf_counter() - started)
            if code == 200:
                text = self._extract(data)
                if schema:
                    if not text.strip():
                        raise RuntimeError("empty output — raise max_output_tokens")
                    return parse_json_lenient(text)
                return text
            refused = self._refused_param(err(data))
            if refused == "service_tier" and self.tier != "default":
                self.tier = "default"
                continue
            if refused in sampling:
                sampling.pop(refused)
                continue
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        raise RuntimeError("request still refused after dropping every tunable")


bot = ProductionClient(MODEL, project=project_id, tier="flex")

Exercise it, then read the metrics it recorded.

In [18]:
print("plain     :", bot.ask("Name one benefit of fair-share scheduling.")[:120])
print(
    "structured:",
    bot.ask("Describe the Rust language.", schema=SCHEMA, max_output_tokens=400),
)
print("metrics   :", bot.metrics.report())

# The routing fix, exercised: the same class against a Chat-Completions-only model
# and a Messages-only model. Both 400ed in the previous version of this notebook.
for other in ("qwen.qwen3-32b", "anthropic.claude-haiku-4-5"):
    probe = ProductionClient(other, project=project_id, tier="flex")
    answer = probe.ask("Name one benefit of queues. One sentence.",
                       max_output_tokens=120, temperature=0.7, top_p=0.95)
    print(f"{other:28} [{probe.surface}] {' '.join(answer.split())[:60]}")

plain     : One primary benefit of fair-share scheduling is that it **prevents a single user or group from monopolizing system resou


structured: {'language': 'Rust', 'typed': True}
metrics   : {'ok': 2, 'throttled': 0, 'server_error': 0, 'client_error': 0, 'timeout': 0} | p50=1.25s p95=1.25s n=2


qwen.qwen3-32b               [chat] Queues ensure orderly processing by maintaining the sequence


anthropic.claude-haiku-4-5   [messages] Queues enable fair, orderly processing of requests by ensuri


## 9b. Guardrails — not a `bedrock-mantle` feature, and the trap that hides it

Amazon Bedrock Guardrails is the service's content-safety control: denied topics,
content filters, word filters, PII redaction, contextual grounding. Every other
control in this notebook is reachable from `bedrock-mantle`. This one is not, and
the way it fails is worth more than the fact itself.

| How you might attach a guardrail | What happens |
|---|---|
| `guardrailConfig` in a `bedrock-mantle` request body | **400** `Unknown parameter: 'guardrailConfig'` |
| `guardrailConfig` in a `bedrock-runtime` `/openai/v1` body | **400** `Unknown parameter` |
| `X-Amzn-Bedrock-GuardrailIdentifier` header on `bedrock-runtime` `/openai/v1` | **200 — and silently ignored** |
| `guardrailConfig` on Converse | works — `stopReason=guardrail_intervened` |
| `guardrailIdentifier` on `InvokeModel` | works |
| `ApplyGuardrail` called directly | works, from any endpoint |

**The third row is the dangerous one.** The header is accepted, the request
succeeds, and the guardrail does nothing. A team that sets it and sees HTTP 200
has no protection and no error to tell them so. We verified this with a guardrail
that denies investment advice: with the header attached, the model returned a
direct buy recommendation.

So there are exactly two supported shapes:

1. **Use Converse on `bedrock-runtime`** and pass `guardrailConfig`. This is the
   canonical path and the one AWS documents.
2. **Call `ApplyGuardrail` yourself** — a standalone `bedrock-runtime` API that
   evaluates text against a guardrail and returns a verdict. It takes no model and
   costs no inference, so it works as a pre-filter on input and a post-filter on
   output *even for a `bedrock-mantle` workload*. That is the answer if you need
   both mantle and guardrails.

The cells below create a throwaway guardrail, demonstrate both shapes plus the
silent-header trap, and delete it again.

In [19]:
import boto3

control = boto3.client("bedrock", region_name=REGION)
runtime = boto3.client("bedrock-runtime", region_name=REGION)

# A deliberately narrow guardrail so the verdict is unambiguous. Created here and
# deleted in the last cell of this section -- like the demo project above, this
# notebook cleans up what it makes.
guardrail = control.create_guardrail(
    name="mantle-samples-hardening-demo",
    description="Throwaway guardrail for the per-model Bedrock samples. Safe to delete.",
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "InvestmentAdvice",
                "definition": "Specific recommendations to buy or sell securities.",
                "examples": ["Should I buy AMZN stock?"],
                "type": "DENY",
            }
        ]
    },
    contentPolicyConfig={
        "filtersConfig": [
            {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"}
        ]
    },
    blockedInputMessaging="Blocked by guardrail (input).",
    blockedOutputsMessaging="Blocked by guardrail (output).",
)
GUARDRAIL_ID, GUARDRAIL_VERSION = guardrail["guardrailId"], guardrail["version"]
safe_print("created guardrail:", GUARDRAIL_ID, "version", GUARDRAIL_VERSION)

# A new guardrail takes a moment to become READY.
for _ in range(20):
    if control.get_guardrail(
        guardrailIdentifier=GUARDRAIL_ID, guardrailVersion=GUARDRAIL_VERSION
    )["status"] == "READY":
        break
    time.sleep(2)
print("status:", control.get_guardrail(
    guardrailIdentifier=GUARDRAIL_ID, guardrailVersion=GUARDRAIL_VERSION)["status"])

created guardrail: xzwjhfnc50kr version DRAFT


status: READY


In [20]:
DENIED = "Should I buy AMZN stock right now? Give me a direct recommendation."
ALLOWED = "What is the difference between an index fund and an ETF?"

# Shape 2 first, because it is the one that works for a mantle workload: evaluate
# the text yourself, then decide whether to call the model at all.
print("ApplyGuardrail — a pre-filter you can run before any endpoint")
print("-" * 66)
for text in (DENIED, ALLOWED):
    verdict = runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        source="INPUT",
        content=[{"text": {"text": text}}],
    )
    topics = [
        t["name"]
        for assessment in verdict.get("assessments", [])
        for t in assessment.get("topicPolicy", {}).get("topics", [])
    ]
    print(f"  {verdict['action']:22} topics={topics}  {text[:40]}")

print()
print("=> action=GUARDRAIL_INTERVENED means do not send it. Run the same call with")
print("   source='OUTPUT' on the model's reply to screen what you return.")

ApplyGuardrail — a pre-filter you can run before any endpoint
------------------------------------------------------------------


  GUARDRAIL_INTERVENED   topics=['InvestmentAdvice']  Should I buy AMZN stock right now? Give 


  NONE                   topics=[]  What is the difference between an index 

=> action=GUARDRAIL_INTERVENED means do not send it. Run the same call with
   source='OUTPUT' on the model's reply to screen what you return.


In [21]:
# Shape 1: guardrailConfig on Converse. Works across providers -- Nova, Claude and
# GPT-5.6 all honour it, and the stop reason names what happened.
print("Converse with guardrailConfig")
print("-" * 66)
# Note the model IDs: `anthropic.claude-sonnet-5` is in both catalogues, whereas
# the mantle ID `anthropic.claude-haiku-4-5` has no bedrock-runtime entry under
# that name at all -- another reason to resolve IDs rather than reuse them.
for model in ("amazon.nova-micro-v1", "anthropic.claude-sonnet-5"):
    try:
        reply = runtime.converse(
            modelId=resolve_runtime_id(model, REGION),
            messages=[{"role": "user", "content": [{"text": DENIED}]}],
            inferenceConfig={"maxTokens": 120},
            guardrailConfig={
                "guardrailIdentifier": GUARDRAIL_ID,
                "guardrailVersion": GUARDRAIL_VERSION,
            },
        )
        text = "".join(
            b.get("text", "") for b in reply["output"]["message"]["content"]
        )
        print(f"  {model:28} stop={reply['stopReason']:22} {text[:34]!r}")
    except Exception as exc:  # noqa: BLE001 - report, do not stop the notebook
        print(f"  {model:28} {type(exc).__name__}: {str(exc)[-60:]}")

# And the trap. The header is accepted and does nothing.
print()
print("The silent-header trap on bedrock-runtime's OpenAI APIs")
print("-" * 66)
signed_base = f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1"
for label, extra_headers in (
    ("no guardrail header  ", {}),
    ("WITH guardrail header", {
        "X-Amzn-Bedrock-GuardrailIdentifier": GUARDRAIL_ID,
        "X-Amzn-Bedrock-GuardrailVersion": GUARDRAIL_VERSION,
    }),
):
    payload = json.dumps({
        # GPT-5.6 on bedrock-runtime is inference-profile-only, so name the profile.
        "model": "us.openai.gpt-5.6-sol",
        "input": DENIED,
        "max_output_tokens": 200,
    })
    request = AWSRequest(
        method="POST", url=f"{signed_base}/responses", data=payload,
        headers={"Content-Type": "application/json", **extra_headers},
    )
    SigV4Auth(
        boto3.Session(region_name=REGION).get_credentials().get_frozen_credentials(),
        "bedrock", REGION,
    ).add_auth(request)
    reply = URLLib3Session(timeout=120).send(request.prepare())
    body = json.loads(reply.text) if reply.text.strip() else {}
    blocked = "Blocked by guardrail" in reply.text
    answer = " ".join(response_text(body).split())[:52]
    print(f"  {label} HTTP {reply.status_code} blocked={blocked!s:5} {answer!r}")

print()
print("=> Both calls answered. The header is not an error and not a control:")
print("   it is ignored. Guardrail a GPT model through Converse, or screen the")
print("   text yourself with ApplyGuardrail.")

Converse with guardrailConfig
------------------------------------------------------------------


  amazon.nova-micro-v1         stop=guardrail_intervened   'Blocked by guardrail (input).'


  anthropic.claude-sonnet-5    stop=guardrail_intervened   'Blocked by guardrail (input).'

The silent-header trap on bedrock-runtime's OpenAI APIs
------------------------------------------------------------------


  no guardrail header   HTTP 200 blocked=False '**Direct recommendation: Buy AMZN gradually, not as '


  WITH guardrail header HTTP 200 blocked=False '**Direct recommendation: Don’t buy a full AMZN posit'

=> Both calls answered. The header is not an error and not a control:
   it is ignored. Guardrail a GPT model through Converse, or screen the
   text yourself with ApplyGuardrail.


In [22]:
# Delete the throwaway guardrail. Only the one this notebook created.
control.delete_guardrail(guardrailIdentifier=GUARDRAIL_ID)
remaining = [g["name"] for g in control.list_guardrails().get("guardrails", [])]
safe_print("deleted mantle-samples-hardening-demo | guardrails left in account:",
           len(remaining))

deleted mantle-samples-hardening-demo | guardrails left in account: 2


## 10. Pre-launch checklist

Run through this before your first real traffic.

In [23]:
CHECKLIST = [
    ("Short-term tokens minted in-process, TTL <= 15 min", True),
    ("No long-term API keys anywhere in the deployment", True),
    ("Retry with exponential backoff on 429 and 5xx", True),
    ("Fail fast (no retries) on other 4xx", True),
    ("Client-side timeout on every call", True),
    ("Concurrency ramps over minutes, not seconds", True),
    ("store=False unless server-side state is required", True),
    ("data_retention mode chosen deliberately (none for regulated data)", True),
    ("Every workload runs under a tagged Project", True),
    ("Dashboards point at AWS/BedrockMantle, not AWS/Bedrock", True),
    ("5xx tracked client-side (no server metric exists)", True),
    ("Structured output parsed leniently and validated", True),
    ("Sampling params gated per model", True),
    ("service_tier gated per model", True),
    ("Region pre-flight check for every required model", True),
    ("Quota escalation path known (Support case, not Service Quotas)", True),
    ("Guardrails applied via Converse or ApplyGuardrail, not a header", True),
]
width = max(len(item) for item, _ in CHECKLIST)
for item, done in CHECKLIST:
    print(f"  [{'x' if done else ' '}] {item:{width}}")
print(
    f"\n{sum(1 for _, d in CHECKLIST if d)}/{len(CHECKLIST)} controls implemented "
    f"in this notebook"
)

  [x] Short-term tokens minted in-process, TTL <= 15 min               
  [x] No long-term API keys anywhere in the deployment                 
  [x] Retry with exponential backoff on 429 and 5xx                    
  [x] Fail fast (no retries) on other 4xx                              
  [x] Client-side timeout on every call                                
  [x] Concurrency ramps over minutes, not seconds                      
  [x] store=False unless server-side state is required                 
  [x] data_retention mode chosen deliberately (none for regulated data)
  [x] Every workload runs under a tagged Project                       
  [x] Dashboards point at AWS/BedrockMantle, not AWS/Bedrock           
  [x] 5xx tracked client-side (no server metric exists)                
  [x] Structured output parsed leniently and validated                 
  [x] Sampling params gated per model                                  
  [x] service_tier gated per model                              

In [24]:
code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("cleaned up demo project:", code, archived.get("status"))

cleaned up demo project: 200 archived


## Gotchas — production on bedrock-mantle

| Gotcha | Detail |
|---|---|
| Token lifetime | ≤12 h, **not refreshable**, Region-pinned — mint and cache |
| No RPM quota | Token-based throttling only; most models have no published TPM |
| Quota increases | AWS Support case, **not** the Service Quotas console |
| Wrong path may stall | Always set a client timeout; bound your retries |
| `store` defaults to true | 30-day retention unless you opt out per request |
| Retention gating | A model can be `unavailable` under your retention mode |
| CloudWatch namespace | `AWS/BedrockMantle` — the wrong one shows zero errors |
| No 5xx metric | Track server errors from client-side telemetry |
| CloudTrail | Inference = data events (opt-in, extra cost) |
| Short-term key minting | Never logged — client-side by design |
| 200 ≠ correct | Truncation, trailing characters, ignored constraints all give 200 |
| Per-model params | Sampling and tier support vary — gate them |
| No CRIS / PT / batch | Cross-Region, Provisioned Throughput and batch are runtime-only |
| **No guardrails on mantle** | Not a parameter here. The `bedrock-runtime` OpenAI-API *header* is accepted and **silently ignored** — use Converse or `ApplyGuardrail` (§9b) |
| Routing, not just prefixes | A client must resolve the API surface as well as the path, or it 400s on every Claude and Chat-Completions-only model |

## Where next
- `01-choosing-a-model-and-api.ipynb` — the live capability survey
- `02-migrating-from-openai.ipynb` — porting an existing codebase
- `../00-foundations/` — the underlying mechanics in depth